In [1]:
import sys
sys.path.append('..')

import os
from sqlalchemy import create_engine, text
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, Polygon, MultiPolygon, MultiPoint
from shapely import wkb
from geopandas.tools import sjoin

from utilities_amigocloud import AmigocloudFunctions

from config import RUTA_UNIDAD_ONE_DRIVE
from config import RUTA_LOCAL_ONE_DRIVE
from config import API_AMIGOCLOUD_TOKEN_ADM
from config import POSTGRES_UTEA

RUTA_COMPLETA = os.path.join(RUTA_UNIDAD_ONE_DRIVE, RUTA_LOCAL_ONE_DRIVE)
#PATH_CAT = RUTA_UNIDAD_ONE_DRIVE + r'\Ingenio Azucarero Guabira S.A\UTEA - SEMANAL - EQUIPO AVIACION UTEA\Pulverizacion\2025\SHP\catastro_S09_MIERCOLES.shp'
PATH_CAT = ''
PATH_XLSX_GRUPOS = RUTA_UNIDAD_ONE_DRIVE + r'\Ingenio Azucarero Guabira S.A\UTEA - SEMANAL - AVANCE COSECHA\2025\DATA\GRUPO_COSECHA.xlsx'

In [2]:
def obtener_engine():
    return create_engine(
        f"postgresql+psycopg2://{POSTGRES_UTEA['USER']}:{POSTGRES_UTEA['PASSWORD']}@{POSTGRES_UTEA['HOST']}:{POSTGRES_UTEA['PORT']}/{POSTGRES_UTEA['DATABASE']}"
    )

def obtener_planificacion():
    engine = obtener_engine()
    try:
        query = """
            SELECT *
            FROM drones_pulverizacion.planificacion_pulv
            WHERE procesado=false;
        """
        gdf = gpd.read_postgis(query, engine, geom_col='geom')
        return gdf
    except Exception as e:
        print(f"Error en la consulta: {e}")
        return gpd.GeoDataFrame()
    return None

def marcar_como_procesado(id_os):
    engine = obtener_engine()  # tu función que crea el engine
    try:
        with engine.begin() as conn:
            query = text("""
                UPDATE drones_pulverizacion.planificacion_pulv
                SET procesado = true
                WHERE id = :id_os
            """)
            conn.execute(query, {"id_os": id_os})
            print(f"✔️ id {id_os} marcado como procesado.")
    except Exception as e:
        print(f"❌ Error al actualizar: {e}")
    return None

def marcar_lote_como_planificado(id_lote):
    engine = obtener_engine()  # tu función que crea el engine
    try:
        with engine.begin() as conn:
            query = text("""
                UPDATE drones_pulverizacion.parte_diario_pulv SET estado = 'PLANIFICADO' WHERE id = :id_lote
            """)
            conn.execute(query, {"id_lote": id_lote})
            print(f"✔️ lote id={id_lote} planificado.")
    except Exception as e:
        print(f"❌ Error al actualizar: {e}")
    return None

def obtener_parte_diario_sin_planificar():
    engine = obtener_engine()
    try:
        query = f"""
            SELECT * FROM drones_pulverizacion.parte_diario_pulv where estado is null
        """
        gdf = gpd.read_postgis(query, engine, geom_col='geom')
        return gdf
    except Exception as e:
        print(f"❌ No se pudo obtener parte_diario sin planificar. Error en la consulta: {e}")
        return gpd.GeoDataFrame()
    return None

def convertir_a_multipolygon(geometry):
    if isinstance(geometry, Polygon):
        return MultiPolygon([geometry])
    return geometry

def convertir_a_wkb(polygon):
    wkb_data = wkb.dumps(polygon, hex=True)
    return wkb_data

# FUNCION DEPRECIADAD, SE OPTO POR CARGA MASIVA
def cargar_a_amigocloud(gdf):
    # repreyectar a WGS84
    gdf_pla_gral = gdf.to_crs(epsg=4326)
    # convertir poligonos a multipoligonos
    gdf_pla_gral['geom'] = gdf_pla_gral['geom'].apply(convertir_a_multipolygon)
    
    gdf_pla_gral['unidad_01'] = gdf_pla_gral['unidad_01'].astype(int)
    gdf_pla_gral['unidad_03'] = gdf_pla_gral['unidad_03'].astype(int)
    gdf_pla_gral['os'] = gdf_pla_gral['os'].astype(int)
    gdf_pla_gral['soca'] = gdf_pla_gral['soca'].astype(int)
    gdf_pla_gral['id'] = gdf_pla_gral['id'].astype(int)
    gdf_pla_gral['inst'] = gdf_pla_gral['inst'].astype(int)
    gdf_pla_gral['mezcla'] = gdf_pla_gral['mezcla'].astype(int)
    
    # recorrer el gdf de lotes y cargarlo a amigocloud
    id_proyecto = 35248
    for index, row in gdf_pla_gral.iterrows():
        wkb_hex = convertir_a_wkb(row['geom'])
        insert_sql = f"""
            INSERT INTO dataset_360912 (id, unidad_01, unidad_02, unidad_03, unidad_04, unidad_05, area, os, mezcla, geometry)
            VALUES ({row['id']}, {row['unidad_01']}, '{row['unidad_02']}', {row['unidad_03']}, '{row['unidad_04']}', '{row['unidad_05']}', {row['area']}, '{row['os']}', '{row['mezcla']}', ST_SetSRID(ST_GeomFromWKB('\\x{wkb_hex}'), 4326));
        """
        query_sql = {'query': insert_sql}
        resultado_post = amigocloud.ejecutar_query_sql(id_proyecto, insert_sql, 'post')
        print(f'Lote registrado en AmigoCloud')
        marcar_lote_como_planificado(row['id'])
    return None

def get_catastro():
    engine = obtener_engine()
    try:
        query = f'''
            SELECT * FROM catastro_iag.catastro
        '''
        gdf = gpd.read_postgis(query, engine, geom_col='geom')
        return gdf
    except Exception as e:
        print(f"❌ Error al obtener la capa de catastro: {e}")
        return gpd.GeoDataFrame()
    return None

def get_catastro_adm():
    engine = obtener_engine()
    try:
        query = f'''
            SELECT * FROM catastro_iag.catastro_adm
        '''
        gdf = gpd.read_postgis(query, engine, geom_col='geom')
        return gdf
    except Exception as e:
        print(f"❌ Error al obtener la capa de catastro: {e}")
        return gpd.GeoDataFrame()
    return None

In [3]:
def cargar_a_amigocloud_bulk(gdf):
    # 1. Preparación de datos (Igual que antes)
    gdf_pla_gral = gdf.to_crs(epsg=4326).copy()
    
    columnas_int = ['unidad_01', 'unidad_03', 'os', 'soca', 'id', 'inst', 'mezcla']
    for col in columnas_int:
        if col in gdf_pla_gral.columns:
            gdf_pla_gral[col] = gdf_pla_gral[col].fillna(0).astype(int)

    # 2. Construir la lista de valores para el SQL
    valores_rows = []
    
    for index, row in gdf_pla_gral.iterrows():
        wkb_hex = convertir_a_wkb(convertir_a_multipolygon(row['geom']))
        
        # Formateamos cada fila como una tupla de SQL (val1, val2, ..., geom)
        fila_sql = f"""(
            {row['id']}, {row['unidad_01']}, '{row['unidad_02']}', 
            {row['unidad_03']}, '{row['unidad_04']}', '{row['unidad_05']}', 
            {row['area']}, {row['os']}, {row['mezcla']}, 
            ST_SetSRID('{wkb_hex}'::geometry, 4326)
        )"""
        valores_rows.append(fila_sql)

    # 3. Unir todas las filas con comas
    todos_los_valores = ",\n".join(valores_rows)

    # 4. Crear el Query Final
    id_proyecto = 35248
    insert_sql = f"""
        INSERT INTO dataset_360912 
        (id, unidad_01, unidad_02, unidad_03, unidad_04, unidad_05, area, os, mezcla, geometry)
        VALUES 
        {todos_los_valores};
    """

    # 5. Ejecutar una sola vez
    try:
        print(f"Enviando {len(gdf_pla_gral)} registros a AmigoCloud...")
        resultado = amigocloud.ejecutar_query_sql(id_proyecto, insert_sql, 'post')
        
        # Solo si no hay error en la respuesta, procedemos a marcar
        # Asumo que ejecutar_query_sql lanza una excepción si falla o devuelve algo verificable
        if resultado: 
            print("✅ AmigoCloud aceptó los datos.")
            for l_id in gdf_pla_gral['id']:
                marcar_lote_como_planificado(l_id)
                print(f"✔️ lote id={l_id} marcado en sistema local.")
        else:
            print("❌ La carga falló en el servidor. No se marcaron lotes como planificados.")
        
    except Exception as e:
        print(f"Error en la carga masiva: {e}")

    return None

In [4]:
amigocloud = AmigocloudFunctions(token=API_AMIGOCLOUD_TOKEN_ADM)
amigocloud

# CARGAR LOTES A AMIGOCLOUD

In [5]:
# obtener parte diario sin planificar
parte_diario_sin_planificar = obtener_parte_diario_sin_planificar()
print(parte_diario_sin_planificar['area'].sum(), 'ha. para subir a amigocloud')
# lista de os del parte diario son planificar
lista_os_sin_planificar = list(set(parte_diario_sin_planificar['os']))
print(len(lista_os_sin_planificar), 'ordenes de servicio sin planificar')

23.42544750853223 ha. para subir a amigocloud
1 ordenes de servicio sin planificar


In [6]:
# CARGAR A AMIGOCLOD
# recorriendo la lista de os
for i in lista_os_sin_planificar:
    plan_os = parte_diario_sin_planificar[parte_diario_sin_planificar['os'] == i]
    cargar_a_amigocloud_bulk(plan_os)

Enviando 1 registros a AmigoCloud...
✅ AmigoCloud aceptó los datos.
✔️ lote id=1331 planificado.
✔️ lote id=1331 marcado en sistema local.
